<a href="https://colab.research.google.com/github/SharvChopra/LLM_Code/blob/main/Tokenizer_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
## character level tokenization

class Tokenizer:
  def encode(self,text):
    return [ord(c) for c in text]

  def decode(self,token):
    return "".join(chr(t) for t in token)

token = Tokenizer()


my_text = "Hello, AI!"
print(f"Original Text: '{my_text}'")

# 3. Encode the text (Human -> Machine)
encoded_tokens = token.encode(my_text)
print(f"Encoded Tokens: {encoded_tokens}")

# 4. Decode the tokens (Machine -> Human)
decoded_text = token.decode(encoded_tokens)
print(f"Decoded Text:  '{decoded_text}'")

Original Text: 'Hello, AI!'
Encoded Tokens: [72, 101, 108, 108, 111, 44, 32, 65, 73, 33]
Decoded Text:  'Hello, AI!'


In [4]:
from collections import Counter

class BPE_Tokenizer:
  def __init__(self):
    ## 2 empty dictionaries
    self.merges = {}
    self.vocab = {}

  def _get_pairs(self,tokens):
    pairs = Counter()
    for i in range(len(tokens)-1):
      pairs[(tokens[i], tokens[i + 1])] += 1
    return pairs

  def _merge_pair(self,tokens,pair,new_token):
    merged = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
          merged.append(new_token)
          i += 2
        else:
          merged.append(tokens[i])
          i += 1
    return merged

  def train(self, text, num_merges):
        tokens = list(text.encode("utf-8"))
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            new_token = 256 + i
            tokens = self._merge_pair(tokens, best_pair, new_token)
            self.merges[best_pair] = new_token
            self.vocab[new_token] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

        return self

  def encode(self, text):
        tokens = list(text.encode("utf-8"))
        for pair, new_token in self.merges.items():
            tokens = self._merge_pair(tokens, pair, new_token)
        return tokens

  def decode(self, tokens):
      byte_sequence = b"".join(self.vocab[t] for t in tokens)
      return byte_sequence.decode("utf-8", errors="replace")

In [5]:
corpus = (
    "The cat sat on the mat. The cat ate the rat. "
    "The dog sat on the log. The dog ate the frog. "
    "Natural language processing is the study of how computers "
    "understand and generate human language. "
    "Tokenization is the first step in any NLP pipeline."
)

tokenizer = BPE_Tokenizer()
tokenizer.train(corpus, num_merges=40)

test_sentences = [
    "The cat sat on the mat.",
    "Natural language processing",
    "tokenization pipeline",
    "unhappiness",
]

for sentence in test_sentences:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded)
    raw_bytes = len(sentence.encode("utf-8"))
    ratio = len(encoded) / raw_bytes
    print(f"'{sentence}'")
    print(f"  Tokens: {len(encoded)} (from {raw_bytes} bytes) -- ratio: {ratio:.2f}")
    print(f"  Roundtrip: {'PASS' if decoded == sentence else 'FAIL'}")

'The cat sat on the mat.'
  Tokens: 3 (from 23 bytes) -- ratio: 0.13
  Roundtrip: PASS
'Natural language processing'
  Tokens: 17 (from 27 bytes) -- ratio: 0.63
  Roundtrip: PASS
'tokenization pipeline'
  Tokens: 16 (from 21 bytes) -- ratio: 0.76
  Roundtrip: PASS
'unhappiness'
  Tokens: 10 (from 11 bytes) -- ratio: 0.91
  Roundtrip: PASS


In [6]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

texts = [
    "The cat sat on the mat.",
    "unhappiness",
    "Hello, world!",
    "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
    "Geschwindigkeitsbegrenzung",
]

for text in texts:
    our_tokens = tokenizer.encode(text)
    tiktoken_tokens = enc.encode(text)
    tiktoken_pieces = [enc.decode([t]) for t in tiktoken_tokens]
    print(f"'{text}'")
    print(f"  Our BPE:   {len(our_tokens)} tokens")
    print(f"  tiktoken:  {len(tiktoken_tokens)} tokens -> {tiktoken_pieces}")

'The cat sat on the mat.'
  Our BPE:   3 tokens
  tiktoken:  7 tokens -> ['The', ' cat', ' sat', ' on', ' the', ' mat', '.']
'unhappiness'
  Our BPE:   10 tokens
  tiktoken:  3 tokens -> ['un', 'h', 'appiness']
'Hello, world!'
  Our BPE:   13 tokens
  tiktoken:  4 tokens -> ['Hello', ',', ' world', '!']
'def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)'
  Our BPE:   68 tokens
  tiktoken:  23 tokens -> ['def', ' fibonacci', '(n', '):', ' return', ' n', ' if', ' n', ' <', ' ', '2', ' else', ' fibonacci', '(n', '-', '1', ')', ' +', ' fibonacci', '(n', '-', '2', ')']
'Geschwindigkeitsbegrenzung'
  Our BPE:   24 tokens
  tiktoken:  9 tokens -> ['G', 'esch', 'wind', 'ig', 'ke', 'its', 'beg', 'ren', 'zung']


In [7]:
def analyze_vocabulary(tokenizer, test_texts):
    total_tokens = 0
    total_chars = 0
    token_usage = Counter()

    for text in test_texts:
        encoded = tokenizer.encode(text)
        total_tokens += len(encoded)
        total_chars += len(text)
        for t in encoded:
            token_usage[t] += 1

    print(f"Vocabulary size: {len(tokenizer.vocab)}")
    print(f"Total tokens across all texts: {total_tokens}")
    print(f"Total characters: {total_chars}")
    print(f"Avg tokens per character: {total_tokens / total_chars:.2f}")

    print(f"\nMost used tokens:")
    for token_id, count in token_usage.most_common(10):
        token_bytes = tokenizer.vocab[token_id]
        display = token_bytes.decode("utf-8", errors="replace")
        print(f"  Token {token_id:4d}: '{display}' (used {count} times)")

    unused = [t for t in tokenizer.vocab if t not in token_usage]
    print(f"\nUnused tokens: {len(unused)} out of {len(tokenizer.vocab)}")